## Setting up our second agent, using Strands Agents and A2A on AgentCore Runtime

On previous module, we've launched our first A2a agent and deployed it on AgentCore Runtime.

On this lab, we're going to create a second agent into the solution, which is another AWS expert, but this one will have an internet tool to query latest News about AWS and latest released AWS blogs.

<img src="images/architecture-lab2.png" style="width: 80%;">

So let's get started!

#### Setup

Import required dependencies

In [ ]:
# Import libraries
import os
import json
import requests
import boto3
from boto3.session import Session
from strands.tools import tool

# Get boto session
boto_session = Session()
region = boto_session.region_name

Retrieve Cognito information from previous LAB, so we can add Cognito information in this Agent.

In [ ]:
%store -r

### 1 - Create AWS Blogs and News expert Agent
Let's create our second agent that will be an AWS expert, focused on Blogs and News.

It will query on internet information about AWS Blogs or recent launches on AWS What's new.

Let's generate Python code that will be used for our agent, and lately will be deployed in AgentCore.

In [ ]:
%%writefile agents/strands_aws_blogs_news.py
import logging
import os
from strands_tools.calculator import calculator
from strands import Agent, tool
from strands.multiagent.a2a import A2AServer
import uvicorn
from fastapi import FastAPI

from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException


logging.basicConfig(level=logging.INFO)

# Use the complete runtime URL from environment variable, fallback to local
runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')

logging.info(f"�  Runtime URL: {runtime_url}")

@tool
def internet_search(keywords: str, region: str = "us-en", max_results: int | None = None) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"


system_prompt = """You are an AWS Blog Expert. 
You will use a internet search tool to get updates or news provided by AWS on:

AWS News Blog: https://aws.amazon.com/blogs/aws/
AWS Blogs for Machine Learning: https://aws.amazon.com/blogs/machine-learning/

Key capabilities:
- Search and retrieve information from Web using AWS oficial websites
- Don't get only homepage info, look for inner domains, like 
- Provide clear, accurate answers about question asked (most recent information)

Guidelines:
- Always prioritize official AWS pages as your source of truth
- Provide specific, actionable information when possible
- Include relevant links or references when helpful
- If you're unsure about something, clearly state your limitations
- Focus on being helpful, accurate, and concise in your responses
- Try to simplify/summarize answers to make it faster, small and objective

You have access to internet_search tools to help answer user questions effectively."""

agent = Agent(system_prompt=system_prompt, 
              tools=[internet_search],
              name="AWS Blog/News Agent",
              description="An agent to search on Web latest AWS Blogs and News.",
              callback_handler=None)

host, port = "0.0.0.0", 9000

# Pass runtime_url to http_url parameter AND use serve_at_root=True
a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True  # Serves locally at root (/) regardless of remote URL path complexity
)

app = FastAPI()

@app.get("/ping")
def ping():
    return {"status": "healthy"}

app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)


Create IAM Role

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_BLOG_ROLE_NAME

agent_name="aws_blog_assistant"

execution_role_arn = create_agentcore_runtime_execution_role(AWS_BLOG_ROLE_NAME)

##### Configure server for deployment

Like previous example, we are adding a new protocol in toolkit configuration:

`protocol="A2A"`

This will create this agent to support A2A protocol inside AgentCore

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    entrypoint="agents/strands_aws_blogs_news.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [COGNITO_CLIENT_ID],
            "discoveryUrl": DISCOVERY_URL,
        }
    },
    protocol="A2A"
)

print("Configuration completed:", response)

Launch the agent on AgentCore Runtime

In [ ]:
launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

agent_arn = launch_result.agent_arn

**Check Deployment Status**

Let's wait for the deployment to complete:

In [ ]:
import time

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

print(f"Final status: {status}")

Export variables to be used in next notebooks

In [ ]:
BLOG_AGENT_ID = launch_result.agent_id
BLOG_AGENT_ARN = launch_result.agent_arn
BLOG_AGENT_NAME = agent_name

%store BLOG_AGENT_ID
%store BLOG_AGENT_ARN
%store BLOG_AGENT_NAME

Store ARN on SSM, so it can be used by orchestrator

In [ ]:
from helpers.utils import put_ssm_parameter, SSM_BLOGS_AGENT_ARN

put_ssm_parameter(SSM_BLOGS_AGENT_ARN, BLOG_AGENT_ARN)

#### Invoking A2A agent to test it

Let's repeat same tests that we did in previous notebook, starting by getting Agent Card information, after refreshing the auth token:

In [ ]:
from helpers.utils import reauthenticate_user

bearer_token = reauthenticate_user(
    COGNITO_CLIENT_ID,
    COGNITO_SECRET
)

In [ ]:
from uuid import uuid4
from urllib.parse import quote


session_id = str(uuid4())
print(f"Generated session ID: {session_id}") # temp keeping same session

def fetch_agent_card(session_id):
    # URL encode the agent ARN
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/.well-known/agent-card.json"
    print(url)
    # Generate a unique session ID
    #session_id = str(uuid4())
    #print(f"Generated session ID: {session_id}")

    # Set headers
    headers = {
        'Accept': '*/*',
        'Authorization': f'Bearer {bearer_token}',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
        'X-Amzn-Trace-Id': f'aws_docs_assistant_{session_id}'
    }

    try:
        # Make the request
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parse and pretty print JSON
        agent_card = response.json()
        print(json.dumps(agent_card, indent=2))

        return agent_card

    except requests.exceptions.RequestException as e:
        print(f"Error fetching agent card: {e}")
        return None

In [ ]:
fetch_agent_card(session_id)

Agent Invoke

In [ ]:
import asyncio
import logging
import os
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 300  # set request timeout to 5 minutes

def create_message(*, role: Role = Role.user, text: str) -> Message:
    return Message(
        kind="message",
        role=role,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )

async def send_sync_message(message: str):
    # Get runtime URL from environment variable
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/"
    
    # Generate a unique session ID
    session_id = str(uuid4())
    print(f"Generated session ID: {session_id}")

    # Add authentication headers for AgentCore
    headers = {"Authorization": f"Bearer {bearer_token}",
              'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id}
        
    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, headers=headers) as httpx_client:
        # Get agent card from the runtime URL
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
        agent_card = await resolver.get_agent_card()
        print(agent_card)

        # Agent card contains the correct URL (same as runtime_url in this case)
        # No manual override needed - this is the path-based mounting pattern

        # Create client using factory
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=False,  # Use non-streaming mode for sync response
        )
        factory = ClientFactory(config)
        client = factory.create(agent_card)

        # Create and send message
        msg = create_message(text=message)

        # With streaming=False, this will yield exactly one result
        async for event in client.send_message(msg):
            if isinstance(event, Message):
                logger.info(event.model_dump_json(exclude_none=True, indent=2))
                return event
            elif isinstance(event, tuple) and len(event) == 2:
                # (Task, UpdateEvent) tuple
                task, update_event = event
                logger.info(f"Task: {task.model_dump_json(exclude_none=True, indent=2)}")
                if update_event:
                    logger.info(f"Update: {update_event.model_dump_json(exclude_none=True, indent=2)}")
                return task
            else:
                # Fallback for other response types
                logger.info(f"Response: {str(event)}")
                return event

In [ ]:
result = await send_sync_message("Give me the latest published blog for Bedrock AgentCore?")

Congratulations, you have deployed your second agent, using A2A protocol on Amazon AgentCore Runtime.

Now, let's move to create an orchestrator.